# Building a Reproducible Mental Health Data Ecosystem: The Kilifi County, Kenya FAIR² Model Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) using the `mlcroissant` library. All dataset entities are referenced using their Croissant schema `@id` fields for reproducibility and traceability.

### Dataset Source
Dataset Croissant schema: [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Set the URL to the Croissant schema
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Access the metadata object directly
meta = dataset.metadata

print(f"Dataset name: {meta.name}\nDescription: {meta.description}")

## 2. Data Overview

Review available record sets, their `@id`s, fields, and columns. All entities are referenced by their `@id` for reproducibility.

In [ ]:
# List all record sets, fields, columns, and their @id
print("Record Sets (@id and name):\n---------------------------")
record_sets = dataset._dataset_dict.get('recordSet', [])
if isinstance(record_sets, dict):
    record_sets = [record_sets]  # Make iterable
for rset in record_sets:
    rset_id = rset.get('@id')
    rset_name = rset.get('name', '')
    print(f"  @id: {rset_id} | name: {rset_name}")
    # List fields for each record set
    fields = rset.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("    Fields (@id, name, dataType):")
    for field in fields:
        field_id = field.get('@id')
        fname = field.get('name', '')
        dtype = field.get('dataType', '')
        print(f"      @id: {field_id} | name: {fname} | dataType: {dtype}")
    # List columns for each record set (if present)
    cols = rset.get('column', [])
    if cols:
        if isinstance(cols, dict):
            cols = [cols]
        print("    Columns (@id, name, source):")
        for col in cols:
            col_id = col.get('@id')
            col_name = col.get('name', '')
            src = col.get('source', '')
            print(f"      @id: {col_id} | name: {col_name} | source: {src}")
if not record_sets:
    print("No record sets defined explicitly in the metadata.\n\nAttempting to discover available record sets via ML Croissant API...")
    # Try listing available record set IDs using the Dataset API, fallback if none
    try:
        # Most Croissant datasets place record_set IDs under dataset.record_set_ids
        print("Record set IDs discovered:", dataset.record_set_ids)
    except Exception:
        print("No recordSet available in this schema.")

## 3. Data Extraction

Load data from a specific record set into a Pandas DataFrame for analysis. Use the record set and field `@id`s identified in the previous overview.

In [ ]:
# Fetch list of available record_set IDs (by @id)
if hasattr(dataset, 'record_set_ids'):
    record_sets_ids = dataset.record_set_ids
else:
    # Fallback: extract from _dataset_dict if possible
    record_sets = dataset._dataset_dict.get('recordSet', [])
    if isinstance(record_sets, dict):
        record_sets = [record_sets]
    record_sets_ids = [rset.get('@id') for rset in record_sets]

dataframes = {}
for rset_id in record_sets_ids:
    try:
        records_iter = dataset.records(record_set=rset_id)
        records = list(records_iter)
        if records:
            df = pd.DataFrame(records)
            dataframes[rset_id] = df
            print(f"Loaded DataFrame for record set {rset_id} (shape: {df.shape})")
    except Exception as e:
        print(f"Could not load records for {rset_id}: {e}")

# Show columns and a sample for one (first) record set
if dataframes:
    first_rset = list(dataframes.keys())[0]
    print(f"\nColumns in DataFrame for record set '{first_rset}':")
    print(dataframes[first_rset].columns.tolist())
    display(dataframes[first_rset].head())
else:
    print("No record sets/record data available for extraction.")

## 4. Exploratory Data Analysis (EDA)

Apply basic EDA operations: filtering, normalization, grouping, and outlier removal as appropriate. Use columns referenced by `@id` or name as listed above.


In [ ]:
# Example: filter, normalize and group a numeric field (e.g., 'phq9_total_score')
import numpy as np

if dataframes:
    # Choose the first dataframe for demonstration
    rset_id = list(dataframes.keys())[0]
    df = dataframes[rset_id].copy()

    # Try automatic search for a likely numeric field
    numeric_candidates = [col for col in df.columns if ('score' in col.lower() or 'age' in col.lower() or df[col].dtype in [np.int64, np.float64])]
    numeric_field = None
    if numeric_candidates:
        for col in numeric_candidates:
            try:
                numeric_values = pd.to_numeric(df[col], errors='coerce')
                if numeric_values.notnull().sum() > 0:
                    numeric_field = col
                    break
            except Exception:
                continue

    print(f"Using numeric field for EDA: {numeric_field}")
    if numeric_field:
        # Ensure numeric type
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = np.nanmean(df[numeric_field])
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records in '{rset_id}' where {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} records")
        # Normalize
        mu, sigma = filtered_df[numeric_field].mean(), filtered_df[numeric_field].std()
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mu) / sigma
        print(f"Normalized '{numeric_field}':")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try grouping by another field, e.g., 'gender' or similar
        group_candidates = [col for col in df.columns if col.lower() in ['gender', 'sex', 'village', 'marital_status', 'level_of_education']]
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
    else:
        print("No suitable numeric field found for demonstration.")
else:
    print("Data not loaded. Run the previous cells to extract records into DataFrames.")

## 5. Visualization

Visualize the distribution of a numeric field (e.g., PHQ-9 total score) and relationships with demographic data.

In [ ]:
# Example histogram and box plot
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals() and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[rset_id][numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    # If group_field available, show boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=dataframes[rset_id][group_field], y=dataframes[rset_id][numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No data or suitable numeric field for visualization.")

## 6. Conclusion

- Successfully loaded the Kilifi County FAIR² mental health dataset from the Croissant schema.
- Explored structure: record sets, fields (referenced by `@id`).
- Extracted records into pandas DataFrames and performed basic filtering and normalization.
- Visualized distributions and compared key statistics by demographic field where possible.
- For deeper analysis, refer to the original Croissant schema and documentation for full field definitions and downstream use.
